In [2]:
from classiq import *


In [2]:
# Exercise 0
from classiq import *

@qfunc
def foo(q: Output[QBit]) -> None:
    allocate(q)
    X(q)
    H(q)


@qfunc
def main(q: Output[QBit]) -> None:
    foo(q)


qmod = create_model(main)
qprog = synthesize(qmod)
job = execute(qprog)
job.get_sample_result().parsed_counts

[{'q': 1}: 1040, {'q': 0}: 1008]

In [3]:
# Exercise 1
@qfunc
def foo(qarr: QArray) -> None:
    H(qarr[0])
    CX(qarr[0], qarr[1])


@qfunc
def main(qarr: Output[QArray]) -> None:
    allocate(2, qarr)
    foo(qarr)


qmod = create_model(main)
qprog = synthesize(qmod)
job = execute(qprog)
job.get_sample_result().parsed_counts

[{'qarr': [0, 0]}: 1045, {'qarr': [1, 1]}: 1003]

In [4]:
# Exercise 2

@qfunc
def my_hadamard_transform(q: QArray) -> None:
    repeat(
        count=q.len,
        iteration=lambda index: H(q[index])
    )

@qfunc
def main(q: Output[QArray]) -> None:
    allocate(10, q)
    my_hadamard_transform(q)


qmod = create_model(main)
qprog = synthesize(qmod)
show(qprog)
# job = execute(qprog)
# job.get_sample_result().parsed_counts

Quantum program link: https://platform.classiq.io/circuit/2vm0l9xsAIN6Q9oiNNbQAp1K7mr?login=True&version=0.75.0


In [5]:
# Exercise 3
from typing import List

import numpy as np

from classiq import *

rng = np.random.default_rng(seed=0)
random_matrix = rng.random((4, 4))
qr_unitary, _ = np.linalg.qr(random_matrix)

unitary_matrix = QConstant("unitary_matrix", List[List[float]], qr_unitary.tolist())

@qfunc
def three_times(q: QArray) -> None:
    repeat(
        count=3,
        iteration=lambda index: unitary(unitary_matrix, q[index])
    )
#doesn't work

@qfunc
def pow_3(q: QArray) -> None:
    power(3, lambda: unitary(unitary_matrix, q))



@qfunc
def main(q: Output[QArray]) -> None:
    allocate(2, q)
    pow_3(q)

qmod = create_model(main)
qprog = synthesize(qmod)
show(qprog)

Quantum program link: https://platform.classiq.io/circuit/2vm0lJjmpFUgV76ZNq5BmpPFGOD?login=True&version=0.75.0


In [6]:
# Exercise 4

@qfunc
def my_apply_to_all(operator: QCallable[QBit], q: QArray) -> None:
    repeat(
        count=q.len,
        iteration=lambda i: operator(q[i])
    )

@qfunc
def main(q: Output[QArray]) -> None:
    allocate(10, q)
    my_apply_to_all(H, q)


qmod = create_model(main)
qprog = synthesize(qmod)
show(qprog)

Quantum program link: https://platform.classiq.io/circuit/2vm0ldEQOGP7ve6lKLGVC4zULtN?login=True&version=0.75.0


In [7]:
# Exercise 5a
from classiq.qmod.symbolic import pi

@qfunc
def main(target: Output[QBit]) -> None:
    allocate(target)
    qb = QBit()
    allocate(qb)
    control(qb, lambda: RY(pi/2, target))
    
qmod = create_model(main)
qprog = synthesize(qmod)
show(qprog)

Quantum program link: https://platform.classiq.io/circuit/2vm0lkZ56OAog2rGYrVI5jsBHQz?login=True&version=0.75.0


In [10]:
# Exercise 5b

@qfunc
def main(x: Output[QNum], q: Output[QBit]) -> None:
    allocate(q)
    x |= 9
    control(x == 9, lambda: X(q))



qmod = create_model(main)
qprog = synthesize(qmod)
show(qprog)

Quantum program link: https://platform.classiq.io/circuit/2vm1MZyaUfg0A3uDooS4etFr56n?login=True&version=0.75.0


In [3]:
# Exercise 6
from dataclasses import dataclass

# @dataclass
# class PauliTerm:
#     pauli: CArray[Pauli] #enum
#     coefficient: CReal


@qfunc
def main(q: Output[QArray[QBit]]) -> None:
    allocate(4, q)
        
    suzuki_trotter(
        [
            PauliTerm(pauli=[Pauli.X, Pauli.Z, Pauli.X, Pauli.X], coefficient=0.5),
            PauliTerm(pauli=[Pauli.Y, Pauli.I, Pauli.Z, Pauli.I], coefficient=0.25),
            PauliTerm(pauli=[Pauli.X, Pauli.I, Pauli.Z, Pauli.Y], coefficient=0.5),
        ],
        evolution_coefficient=3,
        repetitions=4,
        order=2,
        qbv=q
    )


qmod = create_model(main)
qprog = synthesize(qmod)
show(qprog)


Quantum program link: https://platform.classiq.io/circuit/2vnBU4klxTo6tOWK2grZt7A8Mki?login=True&version=0.75.0


In [6]:
# Exercise 7a


@qfunc
def main(res: Output[QNum]) -> None:
    x = QNum("x")
    y = QNum("y")
    z = QNum("z")
    x |= 2
    y |= 7
    z |= 1
    res |= x + y

qmod = create_model(main)
qprog = synthesize(qmod)
show(qprog)

Quantum program link: https://platform.classiq.io/circuit/2vnEOTS0z7pxKj3Ff0s9PJZYXNr?login=True&version=0.75.0


In [16]:
# Exercise 7b

@qfunc
def main(res: Output[QNum]) -> None:
    x = QNum(size=2)
    y = QNum(size=3)
    prepare_state([0.5, 0.0, 0.5, 0], 0.0, x)
    prepare_state([0.0, 0.25, 0.25, 0.25, 0.0, 0.0, 0.25, 0.0], 0.0, y)
    res |= x + y

qmod = create_model(main)
qprog = synthesize(qmod)
show(qprog)

KeyboardInterrupt: 

In [22]:
# Exercise 8a

@qfunc
def main(res: Output[QNum]) -> None:
    x = QNum()
    y = QNum()
    z = QNum()
    a = QNum()

    x |= 3
    y |= 5
    z |= 2

    within_apply(lambda: assign(x + y, a), lambda: assign(a + z, res))

qmod = create_model(main)
qprog = synthesize(qmod)
show(qprog)


Quantum program link: https://platform.classiq.io/circuit/2vnJOrOcReF6gYHj23qsKkFtKC7?login=True&version=0.75.0


In [26]:
# Exercise 8b

@qfunc
def main(res: Output[QNum]) -> None:
    w = QNum()
    x = QNum()
    y = QNum()
    z = QNum()
    a = QNum()

    w |= 4
    x |= 3
    y |= 5
    z |= 2

    within_apply(lambda: assign(x + y, a), lambda: assign(a + z, res))

    res += w
qmod = create_model(main)
qmod = set_constraints(
    qmod, Constraints(optimization_parameter=OptimizationParameter.WIDTH)
)
qprog = synthesize(qmod)
show(qprog)


Quantum program link: https://platform.classiq.io/circuit/2vnLMxTudLYMJeJ8MmYKwMdhjOI?login=True&version=0.75.0
